## Tips

- **Start simple** — a single GRU with 64 units is a good first attempt.
  Only add complexity (stacking, dropout, bidirectional) if the simple model
  is clearly underfitting or overfitting.

- **Use many epochs** — RNNs on time series converge slowly. 10 or 20 epochs
  is almost never enough. Use **at least 100 epochs** and let early stopping
  decide when to stop. The best checkpoint is saved automatically by
  `ModelCheckpoint` — you will not miss the optimal point even if you
  train for too long.

- **Watch the train/val gap** — if train MAE is much lower than val MAE
  you are overfitting. Add dropout, reduce `hidden_size`, or increase
  early stopping patience to give regularization more time to work.

- **Use TensorBoard** — run `tensorboard --logdir runs` in your terminal
  to monitor train and val curves in real time. A healthy training curve
  shows both losses decreasing together. A diverging val curve means overfitting.

- **Learning rate matters** — if the loss is not decreasing after the first
  few epochs, try a lower learning rate (`1e-4` instead of `1e-3`).
  Use `ReduceLROnPlateau` to automatically decay the learning rate when
  val MAE stops improving.

- **The sklearn baseline is hard to beat** — `HistGradientBoosting` with
  explicit lag features is a very strong baseline for tabular time series.
  This is not a failure of the RNN — it reflects a fundamental difference
  between the two approaches: tree models get temporal information from
  hand-crafted features, while RNNs must learn it from raw sequences.
  Getting within 10 bikes/hour of the sklearn baseline (< 44 bikes/hour)
  is an excellent result.

In [9]:
import os
import numpy as np
import joblib

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

from torch.utils.tensorboard import SummaryWriter

In [10]:
save_dir = "data/bike_processed"

# --- Load arrays ---
raw_data    = np.load(os.path.join(save_dir, "raw_data.npy"))
counts = np.load(os.path.join(save_dir, "counts.npy"))
#count_stats = np.load(os.path.join(save_dir, "count_stats.npy"))
train_idx   = np.load(os.path.join(save_dir, "train_idx.npy"))
val_idx     = np.load(os.path.join(save_dir, "val_idx.npy"))
test_idx    = np.load(os.path.join(save_dir, "test_idx.npy"))
naive_mae   = np.load(os.path.join(save_dir, "naive_mae.npy"))
preprocessor = joblib.load(os.path.join(save_dir, "preprocessor_rnn.pkl"))

# --- Recover split sizes ---
num_train = len(train_idx)
num_val   = len(val_idx)
num_test  = len(test_idx)

# --- Count stats in the training set ---
count_mean = counts[train_idx].mean()
count_std  = counts[train_idx].std()
#count_mean, count_std = np.mean([train_idx]), np.std([train_idx])
#count_mean  = count_stats[0]
#count_std   = count_stats[1]

# --- Sanity check ---
print(f"raw_data shape:    {raw_data.shape}")
print(f"counts shape: {counts.shape}")
print(f"train/val/test:    {num_train} / {num_val} / {num_test}")
print(f"counts range: {counts.min():.0f} to {counts.max():.0f} bikes/hour")
print(f"\nNaive baseline — Val MAE: {naive_mae[0]:.2f} | Test MAE: {naive_mae[1]:.2f} bikes/hour")

raw_data shape:    (17210, 28)
counts shape: (17210,)
train/val/test:    12047 / 2581 / 2582
counts range: 1 to 977 bikes/hour

Naive baseline — Val MAE: 93.26 | Test MAE: 80.78 bikes/hour


In [11]:
count_norm = (counts - count_mean) / count_std

In [12]:
class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length, sampling_rate, 
                 start_index, end_index, shuffle=False):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate

        # Valid starting indices: each sequence of length sequence_length
        # sampled every sampling_rate steps needs:
        # (sequence_length - 1) * sampling_rate + 1 rows ahead
        self.indices = np.arange(start_index, end_index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Sample every `sampling_rate` steps for `sequence_length` steps
        steps = np.arange(start, start + self.sequence_length * self.sampling_rate, 
                          self.sampling_rate)
        x = self.data[steps]
        # Target is `delay` steps ahead of the sequence start
        y = self.targets[start + delay]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

In [13]:
sampling_rate   = 1    # already hourly
sequence_length = 24   # look back 24 hours
delay           = 1    # predict 1 hour ahead
batch_size      = 256
num_features    = 28 

train_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0, end_index=num_train,
)
val_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train, end_index=num_train + num_val,
)
test_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train + num_val, end_index=len(raw_data) - delay,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# --- Sanity check ---
inputs, targets = next(iter(train_loader))
print(f"Input shape:  {inputs.shape}")   # (256, 24, num_features)
print(f"Target shape: {targets.shape}")  # (256,)
print(f"Target range: {targets.min():.0f} to {targets.max():.0f}")

Input shape:  torch.Size([256, 24, 28])
Target shape: torch.Size([256])
Target range: -1 to 3


In [14]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    total_mae  = 0.0
    n          = 0
    with torch.set_grad_enabled(training):
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            preds = model(inputs)
            loss  = criterion(preds, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            total_mae  += torch.sum(torch.abs(preds - targets)).item()
            n          += inputs.size(0)
    mae_bikes = (total_mae / n) * count_std
    return total_loss / n,  mae_bikes  # MAE already in °C


def get_predictions(model, dataset):
    model.eval()
    all_preds     = []
    all_targets   = []
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    with torch.no_grad():
        for inputs, targets in loader:
            preds = model(inputs.to(device)).cpu().numpy()
            all_preds.append(preds * count_std + count_mean)  # un-normalize predictions
            all_targets.append(targets.numpy())              # targets already in °C
    return np.concatenate(all_preds), np.concatenate(all_targets)

In [15]:
class SimpleLSTMModel(nn.Module):
    def __init__(self, num_features, hidden_size=128):
        super().__init__()
        self.rnn  = nn.LSTM(input_size=num_features, hidden_size=hidden_size, num_layers = 2, dropout=0.2, batch_first=True, bidirectional=True)
        self.head = nn.Linear(hidden_size*2, 1)

    def forward(self, x):
        # out: (batch, seq_len, hidden_size)
        # return_sequences=False equivalent → take last timestep
        #print(x.shape)
        out, _ = self.rnn(x)
        #print(out.shape)
        #print(out[:,-1,:].shape)
        return self.head(out[:, -1, :]).squeeze(-1)

In [16]:
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = SimpleLSTMModel(num_features=num_features).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/jena_simple_rnn")

#callbacks = [
    # ModelCheckpoint("jena_simple_rnn_best.pt", monitor="val_mae"),
# ]
best_val_mae = float("inf")
patience = 10
wait = 0
# --- Training loop ---
epochs = 50
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save(model.state_dict(), "best_lstm_model.pth")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping en epoch {epoch}")
            break


    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae:.2f} | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae:.2f}")

    # for cb in callbacks:
    #     cb.step(metrics, model) if isinstance(cb, ModelCheckpoint) else cb.step(metrics)

writer.close()

Epoch 01 — train loss: 0.5436, train MAE: 82.35 | val loss: 1.0918, val MAE: 109.60
Epoch 02 — train loss: 0.3259, train MAE: 61.84 | val loss: 0.7243, val MAE: 87.53
Epoch 03 — train loss: 0.1764, train MAE: 43.06 | val loss: 0.3724, val MAE: 60.75
Epoch 04 — train loss: 0.0931, train MAE: 32.03 | val loss: 0.2907, val MAE: 53.01
Epoch 05 — train loss: 0.0766, train MAE: 28.40 | val loss: 0.1562, val MAE: 40.50
Epoch 06 — train loss: 0.0610, train MAE: 25.51 | val loss: 0.6209, val MAE: 87.23
Epoch 07 — train loss: 0.0928, train MAE: 31.08 | val loss: 0.1915, val MAE: 46.73
Epoch 08 — train loss: 0.0611, train MAE: 24.75 | val loss: 0.1340, val MAE: 38.00
Epoch 09 — train loss: 0.0551, train MAE: 23.48 | val loss: 0.1450, val MAE: 39.02
Epoch 10 — train loss: 0.0516, train MAE: 22.80 | val loss: 0.1455, val MAE: 37.93
Epoch 11 — train loss: 0.0498, train MAE: 22.38 | val loss: 0.1374, val MAE: 36.65
Epoch 12 — train loss: 0.0501, train MAE: 22.56 | val loss: 0.1444, val MAE: 37.45
Epo

In [17]:
raw_data.shape

(17210, 28)